# 02 Limpieza y preparacion de datos

Tercera iteracion. Este notebook ejecuta el pipeline una sola vez y funciona como capa de explicacion, revision y validacion de artifacts. No entrena modelos.

## 1. Objetivo

Transformar el raw en datasets trazables, separar features, targets e indice, y aplicar guardrails de leakage para la siguiente fase.

## 2. Trazabilidad del codigo

La logica se centraliza en modulos reutilizables para evitar inconsistencias, permitir pruebas unitarias y facilitar la ejecucion del pipeline fuera de Jupyter.

In [ ]:
import pandas as pd

traceability = pd.DataFrame([
    ["Conversion y limpieza", "clean_data", "mini_tpo.data_cleaning", "clean full y modeling", "Tipos, missing y flags"],
    ["Validacion raw", "validation_summary", "mini_tpo.data_validation", "initial_validation_summary.csv", "Control reproducible de esquema"],
    ["Auditoria derivada", "add_audit_columns", "mini_tpo.data_validation", "columnas audit_*", "Reconciliar resultados"],
    ["Disponibilidad", "feature_availability_audit", "mini_tpo.data_audit", "feature_availability_audit.csv", "Prevenir leakage"],
    ["Artifacts seguros", "export_model_artifacts", "mini_tpo.feature_manifest", "features, targets e index", "Separacion fisica de roles"],
    ["Feature manifest", "create_feature_manifest", "mini_tpo.feature_manifest", "feature_manifest.json", "Contrato de variables"],
    ["Pipeline", "run_preparation", "mini_tpo.pipeline", "todos los outputs", "Unica ruta de ejecucion"],
], columns=["resultado_o_tabla", "funcion_responsable", "modulo", "output_generado", "proposito"])
display(traceability)

## 3. Configuracion

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if not (ROOT / "configs" / "project_config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mini_tpo.constants import (
    FEATURES_MODEL_BASE, FEATURES_MODEL_MINIMAL, FEATURES_MODEL_OPTIONAL,
    TARGET_UPLIFT, TARGET_ROI, LEAKAGE_COLUMNS, POST_PROMO_COLUMNS, AUDIT_COLUMNS,
)
from mini_tpo.data_loading import load_config, read_raw_data, read_clean_full, read_modeling_data
from mini_tpo.feature_manifest import validate_no_leakage
from mini_tpo.pipeline import run_preparation

cfg = load_config(ROOT / "configs" / "project_config.yaml")
TABLES_DIR = ROOT / cfg["project"]["tables_dir"]
pd.set_option("display.max_columns", 120)

## 4. Ejecucion unica del pipeline

In [ ]:
# Esta es la unica llamada a run_preparation() en el notebook.
result = run_preparation()
paths = result["paths"]
display(pd.DataFrame({"artifact": paths.keys(), "ruta": [p.relative_to(ROOT) for p in paths.values()], "existe": [p.exists() for p in paths.values()]}))

# A partir de aqui solo se cargan y revisan outputs ya generados.
raw = read_raw_data(cfg)
clean_full = read_clean_full(cfg)
modeling = read_modeling_data(cfg)
features = pd.read_parquet(paths["safe_features"])
targets = pd.read_parquet(paths["safe_targets"])
index = pd.read_parquet(paths["safe_index"])

## 5. Validacion del raw

Esta tabla proviene de `validation_summary()` en `mini_tpo.data_validation` y fue retornada por el pipeline.

In [ ]:
validation = result["validation_summary"]
display(validation)
assert (validation.loc[validation.check.eq("required_column"), "status"] == "ok").all()

## 6. Tratamiento de tipos

In [ ]:
type_comparison = pd.DataFrame({"raw": raw.dtypes.astype(str), "clean_full": clean_full.drop(columns=["row_id"], errors="ignore").dtypes.astype(str)}).fillna("creada en limpieza")
display(type_comparison)

Las conversiones son explicitas y con errores visibles; no se permiten coerciones silenciosas que creen missing.

## 7. Tratamiento de missing

In [ ]:
missing_summary = pd.DataFrame([
    ["flag_secundario raw", int(raw.flag_secundario.isna().sum()), "Se conserva como desconocido"],
    ["flag_secundario clean", int(clean_full.flag_secundario.isna().sum()), "Categoria explicita"],
    ["flag_secundario_missing", int(clean_full.flag_secundario_missing.sum()), "Indicador opcional de calidad"],
], columns=["control", "registros", "decision"])
display(missing_summary)
display(clean_full.flag_secundario.value_counts(dropna=False).to_frame("registros"))

`null` no equivale a cero: imputarlo como ausencia de promocion secundaria mezclaria falta de registro con una decision comercial real.

## 8. Creacion de row_id

In [ ]:
display(clean_full[["row_id", "fecha_inicio_tanda", "id_material", "subcadena"]].head())
assert clean_full.row_id.is_unique
assert features.row_id.is_unique and targets.row_id.is_unique and index.row_id.is_unique

El identificador es secuencial y reproducible para el archivo actual. Permanece estable mientras no cambie el orden del raw; no es una feature predictiva.

## 9. Flags de dominio

In [ ]:
domain_summary = clean_full[["flag_descuento_fuera_dominio", "flag_duracion_fuera_dominio", "flag_fuera_dominio_optimizacion"]].sum().to_frame("registros")
display(domain_summary)
assert not modeling.flag_fuera_dominio_optimizacion.any()

Clean full conserva los casos fuera de dominio para auditoria; modeling los excluye de la fase inicial para reducir extrapolacion.

## 10. Flag del piso de uplift

In [ ]:
print(f"Registros en piso: {clean_full.flag_uplift_en_piso.sum()} ({clean_full.flag_uplift_en_piso.mean():.2%})")
display(pd.read_csv(TABLES_DIR / "uplift_floor_by_sku.csv"))

El valor original no se modifica. El flag permite diagnosticar posible censura o redondeo en modelado.

## 11. Variables de auditoria

In [ ]:
audit_cols = [c for c in clean_full if c.startswith("audit_") or c.startswith("flag_inconsistencia")]
display(clean_full[audit_cols].describe(include="all"))
assert "audit_volumen_incremental_observado" in audit_cols

`audit_volumen_base_tanda` y `audit_volumen_incremental_observado` explican impacto absoluto, pero usan resultados observados o son controles derivados; nunca entran como features.

## 12. Clasificacion de variables

In [ ]:
availability = pd.read_csv(TABLES_DIR / "feature_availability_audit.csv")
roles = pd.read_csv(TABLES_DIR / "model_role_summary.csv")
display(availability); display(roles)

## 13. Supuestos definitivos de disponibilidad

- `volumen_base_sem` es una estimacion disponible antes del inicio y no usa informacion posterior.
- `elasticidad_estimada` es una estimacion interna previa y no usa informacion posterior.
- `roi` es el KPI oficial entregado por Alicorp, valido como target postpromocion y prohibido como predictor.

Estos son supuestos necesarios de la prueba porque no se solicitara informacion adicional.

## 14. Features base

In [ ]:
print("FEATURES_MODEL_BASE =", FEATURES_MODEL_BASE)
assert FEATURES_MODEL_BASE == ["id_material", "subcadena", "factor_descuento", "duracion_dias", "volumen_base_sem", "elasticidad_estimada", "flag_secundario"]
display(features.head())

Volumen base convierte uplift porcentual en unidades; elasticidad aproxima sensibilidad de demanda ante cambios porcentuales de precio.

## 15. Features minimas

In [ ]:
print('FEATURES_MODEL_MINIMAL =', FEATURES_MODEL_MINIMAL)

Este conjunto permitira medir en sensibilidad el valor agregado de volumen base y elasticidad, sin entrenarlo todavia.

## 16. Features opcionales

In [ ]:
print('FEATURES_MODEL_OPTIONAL =', FEATURES_MODEL_OPTIONAL)

`precio_base` es constante por SKU y no tiene interpretacion independiente; `des_marca` es redundante pero util para cold start; `flag_secundario_missing` representa calidad. Ninguna entra al baseline principal.

## 17. Variables excluidas por leakage

In [ ]:
print("TARGET_UPLIFT =", TARGET_UPLIFT)
print("TARGET_ROI =", TARGET_ROI)
print("LEAKAGE_COLUMNS =", LEAKAGE_COLUMNS)
print("POST_PROMO_COLUMNS =", POST_PROMO_COLUMNS)
print("AUDIT_COLUMNS =", AUDIT_COLUMNS)
validate_no_leakage(features.drop(columns="row_id"))

Targets, volumen promocional, ventas, inversion y reconciliaciones de auditoria no estan disponibles al decidir la tanda.

## 18. Construccion de artifacts

In [ ]:
artifact_summary = pd.DataFrame([
    ["model_features_safe", *features.shape, "Predictores autorizados mas row_id"],
    ["model_targets", *targets.shape, "Targets y flags de calidad"],
    ["model_index", *index.shape, "Fecha e identificadores"],
], columns=["artifact", "filas", "columnas", "proposito"])
display(artifact_summary)
assert features.shape == (2048, 8)

## 19. Feature manifest

Se genera mediante `create_feature_manifest()` en `mini_tpo.feature_manifest`.

In [ ]:
manifest = json.loads(paths["feature_manifest"].read_text(encoding="utf-8"))
manifest_df = pd.DataFrame(manifest["variables"])
display(manifest_df)
display(manifest_df.loc[manifest_df.name.isin(["volumen_base_sem", "elasticidad_estimada", "roi"])])

## 20. Log de transformaciones

Se genera mediante `build_cleaning_log()` en `mini_tpo.data_cleaning`.

In [ ]:
cleaning_log = result["cleaning_log"]
display(cleaning_log)

## 21. Validaciones de alineamiento

In [ ]:
assert len(features) == len(targets) == len(index) == 2048
joined = features.merge(targets, on="row_id", validate="one_to_one").merge(index, on="row_id", validate="one_to_one")
assert len(joined) == 2048
assert {TARGET_UPLIFT, TARGET_ROI}.issubset(targets.columns)
assert not {TARGET_UPLIFT, TARGET_ROI}.intersection(features.columns)
print("Join 1:1 validado para", len(joined), "promociones")

## 22. Comparacion entre datasets

In [ ]:
comparison = pd.DataFrame([
    ["Raw", len(raw), raw.shape[1], "Fuente inalterada"],
    ["Clean full", len(clean_full), clean_full.shape[1], "Auditoria completa y reversible"],
    ["Modeling domain", len(modeling), modeling.shape[1], "Dominio 5%-40% y 5-21 dias"],
    ["Features seguras", len(features), features.shape[1], "row_id mas siete predictores"],
], columns=["dataset", "filas", "columnas", "uso"])
display(comparison)
comparison.to_csv(TABLES_DIR / "dataset_comparison.csv", index=False)

## 23. Outputs generados

In [ ]:
display(pd.DataFrame({'output': paths.keys(), 'ruta_relativa': [p.relative_to(ROOT) for p in paths.values()], 'existe': [p.exists() for p in paths.values()]}))

## 24. Limitaciones

- `row_id` depende del orden estable del raw actual.
- ROI se acepta como KPI oficial, pero no se descompone contablemente.
- Volumen base y elasticidad son validos por supuesto explicito, no por auditoria de sus procesos de origen.
- No se imputan targets ni se aplican transformaciones irreversibles a outliers.

## 25. Preparacion para la siguiente fase

Los artifacts quedan listos para feature engineering y un split temporal futuro. Se evaluaran codificacion, transformaciones robustas, interacciones descuento-elasticidad y baseline-duracion, y el valor incremental de las dos features confirmadas. No se ha entrenado ningun modelo ni construido un optimizador.